In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.optimize import minimize, LinearConstraint
import pandas as pd
import numpy as np
#from utils import process_data
import random

class NeuralBanditPHE:
    def __init__(self, catalysts=['Phenol', 'TFE', '1,1,1,3,3,3-Hexafluoro-2-methyl-2-propanol', 'TBAPF6', 'CsPF6', 'TMAPF6', 'Perylene'], alpha=1e-4, beta=0.1,
                uni_file='uni.txt', bivariate_file='bivariate.txt'):
        self.catalysts = catalysts
        # self.df = process_data(catalysts, uni00_file, bivariate_file)
        self.df = pd.read_csv(r"D:\CO2RR-Setup\0-Manuscript\important code in main text\Fold validation\Trial-5\Step1-get the Weight and Bias\clean_data-total-Kobs-final.csv")  
        # Prepare the dataset, in the shape of [CATALYST] + [CO2] + [CURRENT]
        # For example: Phenol, TFE, CO2, TOTAL CURRENT
        self.df = self.df.apply(pd.to_numeric, errors='coerce')
        self.prepare_XY()
        self.alpha, self.beta = alpha, beta
        
        ## Pre-test the performance
        self.nn = self.prepare_nn(self.X_poly, self.y, test=True)
        
    def prepare_XY(self):
        self.X = self.df[self.catalysts + ['CO2']].to_numpy()
        self.y = self.df['Current'].to_numpy()    #remove the log?
        
        self.poly = PolynomialFeatures(degree=4, interaction_only=False, include_bias=False)
        self.X_poly = self.poly.fit_transform(self.X)

    def prepare_nn(self, X, y, test=False):
        nn = MLPRegressor(hidden_layer_sizes=(2048), max_iter=10000, alpha=self.alpha)
        #nn = MLPRegressor(hidden_layer_sizes=(512), max_iter=100, alpha=self.alpha)
        if test:
           X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
           nn.fit(X_train, y_train)
           y_train_pred = nn.predict(X_train)
           y_test_pred = nn.predict(X_test)

        # 限制预测值必须大于 0
           y_train_pred = np.clip(y_train_pred, 0, None)
           y_test_pred = np.clip(y_test_pred, 0, None)

        # 计算 MAE、RMSE 和 MAPE
           train_mae = mean_absolute_error(y_train, y_train_pred)
           test_mae = mean_absolute_error(y_test, y_test_pred)

           train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
           test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

           train_mape = np.mean(np.abs((y_train - y_train_pred) / np.clip(y_train, 1e-5, None))) * 100
           test_mape = np.mean(np.abs((y_test - y_test_pred) / np.clip(y_test, 1e-5, None))) * 100

        # 打印结果
           print(f"Train Metrics: MAE={train_mae:.3f}, RMSE={train_rmse:.3f}, MAPE={train_mape:.2f}%")
           print(f"Test Metrics:  MAE={test_mae:.3f}, RMSE={test_rmse:.3f}, MAPE={test_mape:.2f}%")
        else:
            nn.fit(X, y)
        return nn
            
    def update(self, catalyst, CO2, Current):
        # Expected: catalyst should /1000, CO2 should /100, Current not change
        assert len(catalyst) == len(self.catalyst)
        new_line = [float(x) for x in catalyst] + [float(CO2)] + [float(Current)]
        df.loc[len(df)] = new_line
        self.prepare_XY()
        return
    
    def func(self, catalyst, CO2, p):
        # here we assume the 
        XCO2 = self.poly.transform(np.append(catalyst, CO2).reshape(1, -1))
        XAr = self.poly.transform(np.append(catalyst, 0.0).reshape(1, -1))
        ITot = self.nn.predict(XCO2)         #remove the exp
        IAr = self.nn.predict(XAr)         #remove the exp
        ITot = max(self.nn.predict(XCO2), 1)
        IAr = max(self.nn.predict(XAr) , 1)
        ICO2 = max(ITot - IAr, 1) # make ICO2 always positive
        
        O1 = ICO2 / 20000
        O2 = ICO2 / ITot
        O2 = min(O2, 1)  # Limit O2 to be less than 1
        return p * O1 + (1 - p) * O2, O1, O2
    
    def select_action(self, CO2):
        # Takes the CO2 as a 0-1 scalar, be careful with $\beta$ since it is 
        # perturbing y = e^\beta y (relative magnitude)
        L = len(self.catalysts)
        perturbed_y = np.random.normal(self.y, self.beta) 
        self.nn = self.prepare_nn(self.X_poly, perturbed_y)
        print(perturbed_y)
        
        x_list, y_list = [], []        
        p_list = np.linspace(0, 1, 9) # change this if you need to change the thres   
        for p in p_list:            # Initial guess
            #x0 = [random.uniform(0, 1.0 / L) for _ in self.catalysts]
            x0 = [0.5, 0.5, 0.0, 0.0, 0.0, 0.1, 0]
            # Constraints for the catalyst groups
            constraints = [
                # Phenol + TFE + 1,1,1,3,3,3-Hexafluoro-2-methyl-2-propanol <= 1
                LinearConstraint([1, 1, 1, 0, 0, 0, 0], 0.3, 1),
                # TBAPF6 + CsPF6 + TMAPF6 = 0.1
                LinearConstraint([0, 0, 0, 1, 1, 1, 0], 0.1, 0.1),                
                # Perylene between 0 and 5
                LinearConstraint([0, 0, 0, 0, 0, 0, 1], 0, 5),
                # Ensure all values are within bounds
                LinearConstraint(np.ones(L), 0.4, 6.1)
            ]
            # Bounds for individual catalysts
            bounds = [
                (0.0, 1.0),  # Phenol (0 to 1, step 0.05)
                (0.0, 1.0),  # TFE (0 to 1, step 0.05)
                (0.0, 1.0),  # 1,1,1,3,3,3-Hexafluoro-2-methyl-2-propanol (0 to 1, step 0.05)
                (0.0, 0.1),  # TBAPF6 (0 to 0.1, step 0.005)
                (0.0, 0.1),  # CsPF6 (0 to 0.1, step 0.005)
                (0.0, 0.1),  # TMAPF6 (0 to 0.1, step 0.005)
                (0.0, 5.0)   # Perylene (0 to 5, step 0.5)
            ]
            # Optimize the function
            ret = minimize(
                fun=lambda _x0: -self.func(_x0, CO2, p)[0],
                x0=x0,
                constraints=constraints,
                bounds=bounds
            )
            self.discrete_level = np.array([0.05, 0.05, 0.05, 0.005, 0.005, 0.005, 0.5])
            discrete_x = np.round(ret.x / self.discrete_level) * self.discrete_level
            x_list.append(discrete_x)
        
        # Predict new y values for the generated catalyst configurations
            
        return p_list, np.stack(x_list)

    def predict_and_save(self, output_file='predicted_currents-total-Kobs.csv'):
        predictions = []
        for _, row in self.df.iterrows():
           catalyst = row[self.catalysts].to_numpy()
           CO2 = row['CO2']
           X_input = self.poly.transform(np.append(catalyst, CO2).reshape(1, -1))
           predicted_current = self.nn.predict(X_input)
        
        # 确保裁剪到非负值
           predicted_current = np.clip(predicted_current, 0, None)
           predictions.append(predicted_current[0])  # 如果是数组，取第一个值

        self.df['Predicted_Current'] = predictions
        self.df.to_csv(output_file, index=False)
        print(f"Predictions saved to {output_file}")
        
model = NeuralBanditPHE()
model.predict_and_save('predicted_currents-total-Kobs.csv')


Train Metrics: MAE=2182.135, RMSE=4303.852, MAPE=118.89%
Test Metrics:  MAE=2556.489, RMSE=4812.015, MAPE=145.09%
Predictions saved to predicted_currents-total-Kobs.csv


In [8]:
import numpy as np
import pandas as pd

# 保存 Layer 0 权重
layer0_w = model.nn.coefs_[0]
pd.DataFrame(layer0_w).to_csv("layer0_weights.csv", index=False)

# 保存 Layer 1 权重
layer1_w = model.nn.coefs_[1]
pd.DataFrame(layer1_w).to_csv("layer1_weights.csv", index=False)

# 保存 Layer 0 偏置
layer0_b = model.nn.intercepts_[0]
pd.DataFrame(layer0_b).to_csv("layer0_biases.csv", index=False)

# 保存 Layer 1 偏置
layer1_b = model.nn.intercepts_[1]
pd.DataFrame(layer1_b).to_csv("layer1_biases.csv", index=False)

print("✅ 权重和偏置已保存为 CSV 文件")

✅ 权重和偏置已保存为 CSV 文件
